In [7]:
import os as os

### Return regression for 0_1

In [72]:
import pyfixest as pf
import pandas as pd

# ─── LOAD ───
df = pd.read_csv('../Data/new_car_1_with_fundamental.csv')
df['call_date'] = pd.to_datetime(df['call_date'])
df['time'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)

# ─── WINSORIZE ───
def winsorize(series, lower=0.01, upper=0.99):
    low  = series.quantile(lower)
    high = series.quantile(upper)
    return series.clip(lower=low, upper=high)

for col in ['car_01', 'vol', 'mom', 'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']:
    df[col] = winsorize(df[col])

# ─── DROP MISSING ───
reg_vars = ['car_01', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']
df = df.dropna(subset=reg_vars).copy()
print(f"Sample: {len(df)} obs | {df['symbol'].nunique()} firms | {df['time'].nunique()} periods")

# ─── RUN MODELS ───

# Model 1 — No FE, two-way clustered SE
m1 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 2 — Firm FE only
m2 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 3 — Time FE only
m3 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | time',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 4 — Firm + Time FE
m4 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol + time',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# ─── RESULTS ───
m4.summary()

Sample: 7493 obs | 472 firms | 23 periods
###

Estimation:  OLS
Dep. var.: car_01, Fixed effects: symbol + time
sample: None = all
Inference:  CRV1
Observations:  7491

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| stress_qa     |     -0.011 |        0.002 |    -6.780 |      0.000 | -0.015 |  -0.008 |
| vol           |      0.198 |        0.221 |     0.896 |      0.380 | -0.260 |   0.655 |
| mom           |     -0.014 |        0.006 |    -2.291 |      0.032 | -0.026 |  -0.001 |
| lnmve         |     -0.037 |        0.005 |    -7.053 |      0.000 | -0.048 |  -0.026 |
| bm            |      0.000 |        0.010 |     0.017 |      0.987 | -0.022 |   0.022 |
| UE            |      0.039 |        0.005 |     8.565 |      0.000 |  0.030 |   0.049 |
| POSWORDS      |      0.417 |        0.196 |     2.127 |      0.045 |  0.010 |   0.824 |
| NEGWORDS      |    

/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [73]:
m1.summary()

###

Estimation:  OLS
Dep. var.: car_01
sample: None = all
Inference:  CRV1
Observations:  7493

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      0.042 |        0.012 |     3.617 |      0.002 |  0.018 |   0.067 |
| stress_qa     |     -0.009 |        0.001 |    -6.268 |      0.000 | -0.012 |  -0.006 |
| vol           |      0.350 |        0.099 |     3.548 |      0.002 |  0.146 |   0.555 |
| mom           |     -0.005 |        0.005 |    -1.111 |      0.278 | -0.016 |   0.005 |
| lnmve         |     -0.002 |        0.001 |    -2.393 |      0.026 | -0.004 |  -0.000 |
| bm            |     -0.001 |        0.003 |    -0.409 |      0.686 | -0.008 |   0.005 |
| UE            |      0.031 |        0.004 |     8.051 |      0.000 |  0.023 |   0.039 |
| POSWORDS      |     -0.013 |        0.143 |    -0.088 |      0.930 | -0.309 |   0.284 |
| N

In [4]:
m4.summary()

###

Estimation:  OLS
Dep. var.: car_01, Fixed effects: symbol + time
sample: None = all
Inference:  CRV1
Observations:  6196

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| stress_qa     |     -0.010 |        0.002 |    -5.903 |      0.000 | -0.014 |  -0.007 |
| vol           |      0.209 |        0.218 |     0.959 |      0.349 | -0.247 |   0.665 |
| mom           |     -0.011 |        0.006 |    -1.829 |      0.083 | -0.023 |   0.002 |
| lnmve         |     -0.040 |        0.007 |    -5.740 |      0.000 | -0.054 |  -0.025 |
| bm            |     -0.006 |        0.011 |    -0.499 |      0.624 | -0.029 |   0.018 |
| UE            |      0.035 |        0.004 |     8.069 |      0.000 |  0.026 |   0.044 |
| POSWORDS      |      0.199 |        0.187 |     1.065 |      0.300 | -0.192 |   0.590 |
| NEGWORDS      |     -1.598 |        0.230 |    -6.959 |      

In [ ]:
import pyfixest as pf
import pandas as pd

# ─── LOAD ───
df = pd.read_csv('../Data/new_car_2180_with_fundamental.csv')
df['call_date'] = pd.to_datetime(df['call_date'])
df['time'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)

# ─── WINSORIZE ───
def winsorize(series, lower=0.01, upper=0.99):
    low  = series.quantile(lower)
    high = series.quantile(upper)
    return series.clip(lower=low, upper=high)

for col in ['car_01', 'vol', 'mom', 'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']:
    df[col] = winsorize(df[col])

# ─── DROP MISSING ───
reg_vars = ['car_01', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']
df = df.dropna(subset=reg_vars).copy()
print(f"Sample: {len(df)} obs | {df['symbol'].nunique()} firms | {df['time'].nunique()} periods")

# ─── RUN MODELS ───

# Model 1 — No FE, two-way clustered SE
m1 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 2 — Firm FE only
m2 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 3 — Time FE only
m3 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | time',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# Model 4 — Firm + Time FE
m4 = pf.feols(
    'car_01 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol + time',
    data=df,
    vcov={'CRV1': 'symbol + time'}
)

# ─── RESULTS ───
m4.summary()

### performance regression 

In [15]:
print(os.getcwd())

/Users/yiqisun/Documents/Thesis/NLP project/regression


In [40]:
performance_data = pd.read_csv("../Data/merged_ROA_OCF.csv")

In [41]:
performance_data.shape

(7505, 36)

In [42]:
performance_data.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole', 'car_01', 'vol', 'mom', 'POSWORDS', 'NEGWORDS', 'lnmve',
       'bm', 'atq', 'UE', 'actual_eps', 'median_forecast', 'numest', 'costat',
       'curcdq', 'datafmt', 'indfmt', 'consol', 'tic', 'datadate', 'gvkey',
       'fqtr', 'fyearq', 'dpq', 'niq', 'wcapq', 'oancfy', 'ocf_q', 'roa_q',
       'future_rank'],
      dtype='str')

In [43]:
performance_data = performance_data[['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole', 'car_01', 'vol', 'mom', 'POSWORDS', 'NEGWORDS', 'lnmve',
       'bm', 'atq', 'UE', 'actual_eps', 'median_forecast', 'numest', 'ocf_q', 'roa_q']]

In [44]:
performance_data.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole', 'car_01', 'vol', 'mom', 'POSWORDS', 'NEGWORDS', 'lnmve',
       'bm', 'atq', 'UE', 'actual_eps', 'median_forecast', 'numest', 'ocf_q',
       'roa_q'],
      dtype='str')

In [45]:
performance_data.shape

(7505, 21)

In [46]:
regress_data = performance_data.dropna(subset=[ 'stress_qa', 'vol', 'mom',
              'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS', 'roa_q', 'ocf_q'])

In [48]:
regress_data.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole', 'car_01', 'vol', 'mom', 'POSWORDS', 'NEGWORDS', 'lnmve',
       'bm', 'atq', 'UE', 'actual_eps', 'median_forecast', 'numest', 'ocf_q',
       'roa_q'],
      dtype='str')

In [49]:
regress_data.groupby('symbol').size()

symbol
A       17
AAL     16
AAPL    18
ABBV    16
ABNB    15
        ..
XYL     18
YUM     17
ZBH      6
ZBRA    15
ZTS     18
Length: 442, dtype: int64

In [51]:
print(regress_data[['symbol', 'call_date', 'roa_q']].head(10))

  symbol   call_date     roa_q
0      A  2020-02-18  0.003789
1      A  2020-05-21  0.007093
2      A  2020-08-18  0.044039
3      A  2021-02-16  0.004979
4      A  2021-05-25  0.002925
5      A  2021-11-22 -0.003272
6      A  2022-02-22  0.006209
7      A  2022-05-24  0.005329
8      A  2022-08-16  0.046898
9      A  2022-11-21 -0.003593


In [52]:
# ─── CREATE TIME VARIABLE ───
regress_data['time'] = (regress_data['year'] * 10 + regress_data['quarter']).astype(int)

# ─── WINSORIZE ───
def winsorize(series, lower=0.01, upper=0.99):
    low  = series.quantile(lower)
    high = series.quantile(upper)
    return series.clip(lower=low, upper=high)

for col in ['roa_q', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']:
    if col in regress_data.columns:
        regress_data[col] = winsorize(regress_data[col])

# ─── DROP MISSING ───
reg_vars = ['roa_q', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS',
            'symbol', 'time']

regress_data = regress_data.dropna(subset=reg_vars).copy()
print(f"\nRegression sample: {len(regress_data)} obs")
print(f"Unique firms:      {regress_data['symbol'].nunique()}")
print(f"Unique periods:    {regress_data['time'].nunique()}")

# ─── MODELS ───

# Model 1 — No FE
m1 = pf.feols(
    'roa_q ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

# Model 2 — Firm FE
m2 = pf.feols(
    'roa_q ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

# Model 3 — Time FE
m3 = pf.feols(
    'roa_q ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | time',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

# Model 4 — Firm + Time FE
m4 = pf.feols(
    'roa_q ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol + time',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

# ─── RESULTS ───
print("\n=== ROA Regression Results ===")
pf.etable([m1, m2, m3, m4],
          labels={'roa_q':    'ROA (Q+2)',
                  'stress_qa': 'Stress QA',
                  'vol':       'VOL',
                  'mom':       'MOM',
                  'lnmve':     'LNMVE',
                  'bm':        'BM',
                  'UE':        'UE',
                  'POSWORDS':  'POSWORDS',
                  'NEGWORDS':  'NEGWORDS'})


Regression sample: 6579 obs
Unique firms:      442
Unique periods:    22


/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 4 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 4 singleton fixed effect(s) dropped from the model.
  warnings.warn(



=== ROA Regression Results ===


GT(_tbl_data=   __index_level_0__ __index_level_1__                          0  \
0               coef         Stress QA  -0.000119 <br> (0.000278)   
1               coef               VOL          0.04 <br> (0.137)   
2               coef               MOM         0.005 <br> (0.003)   
3               coef             LNMVE  -0.000216 <br> (0.000338)   
4               coef                BM        -0.008 <br> (0.003)   
5               coef                UE      0.000488 <br> (0.002)   
6               coef          POSWORDS         0.019 <br> (0.058)   
7               coef          NEGWORDS         0.068 <br> (0.129)   
8               coef         Intercept         0.012 <br> (0.007)   
9                 fe              time                          -   
10                fe            symbol                          -   
11                fe           symbol                           -   
12                fe              time                          -   
13             stats      Observations                      6,579   
14             stats                R²                      0.018   

                           1                          2  \
0   -0.00002 <br> (0.000528)  -0.000269 <br> (0.000245)   
1        -0.224 <br> (0.196)         0.139 <br> (0.048)   
2         0.005 <br> (0.003)         0.006 <br> (0.002)   
3        -0.009 <br> (0.004)   0.000096 <br> (0.000343)   
4        -0.003 <br> (0.007)        -0.009 <br> (0.003)   
5      0.000009 <br> (0.002)      0.000194 <br> (0.001)   
6        -0.172 <br> (0.107)         0.092 <br> (0.047)   
7         0.111 <br> (0.172)         0.076 <br> (0.088)   
8                                                         
9                          -                          -   
10                         x                          -   
11                         -                          -   
12                         -                          x   
13                     6,575                      6,579   
14                     0.102                      0.506   

                            3  
0   -0.000157 <br> (0.000368)  
1          0.044 <br> (0.069)  
2          0.004 <br> (0.002)  
3         -0.003 <br> (0.002)  
4          -0.01 <br> (0.006)  
5      -0.000626 <br> (0.001)  
6          0.021 <br> (0.035)  
7          0.062 <br> (0.079)  
8                              
9                           x  
10                          -  
11                          x  
12                          -  
13                      6,575  
14                      0.571  , _body=<great_tables._gt_data.Body object at 0x175a46fd0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x16171e790>, _spanners=Spanners([SpannerInfo(spanner_id='ROA (Q+2)', spanner_level=1, spanner_label='ROA (Q+2)', spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x16171fb10>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x16917f6d0>, _source_notes=['Format of coefficient cell: Coefficient   (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt

In [53]:
m4.summary()

###

Estimation:  OLS
Dep. var.: roa_q, Fixed effects: symbol + time
sample: None = all
Inference:  CRV1
Observations:  6575

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| stress_qa     |     -0.000 |        0.000 |    -0.425 |      0.675 | -0.001 |   0.001 |
| vol           |      0.044 |        0.069 |     0.637 |      0.531 | -0.099 |   0.186 |
| mom           |      0.004 |        0.002 |     2.009 |      0.058 | -0.000 |   0.008 |
| lnmve         |     -0.003 |        0.002 |    -1.799 |      0.086 | -0.007 |   0.000 |
| bm            |     -0.010 |        0.006 |    -1.615 |      0.121 | -0.022 |   0.003 |
| UE            |     -0.001 |        0.001 |    -0.580 |      0.568 | -0.003 |   0.002 |
| POSWORDS      |      0.021 |        0.035 |     0.607 |      0.551 | -0.051 |   0.094 |
| NEGWORDS      |      0.062 |        0.079 |     0.786 |      0

### Car 2_7 regression 

In [66]:
car_2_7 = pd.read_csv('../Data/new_CAR_7_with_fundamental.csv')
car_2_7.shape

(8022, 21)

In [67]:
car_2_7.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole_x', 'car_27', 'stress_whole_y', 'car_01', 'vol', 'mom',
       'POSWORDS', 'NEGWORDS', 'lnmve', 'bm', 'atq', 'UE', 'actual_eps',
       'median_forecast', 'numest'],
      dtype='str')

In [68]:
car_2_7.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole_x', 'car_27', 'stress_whole_y', 'car_01', 'vol', 'mom',
       'POSWORDS', 'NEGWORDS', 'lnmve', 'bm', 'atq', 'UE', 'actual_eps',
       'median_forecast', 'numest'],
      dtype='str')

In [70]:
car_2_7['time'] = (car_2_7['year'] * 10 + car_2_7['quarter']).astype(int)

# ─── WINSORIZE ───
def winsorize(series, lower=0.01, upper=0.99):
    low  = series.quantile(lower)
    high = series.quantile(upper)
    return series.clip(lower=low, upper=high)

for col in ['car_27', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS']:
    if col in df.columns:
        df[col] = winsorize(df[col])

# ─── DROP MISSING ───
reg_vars = ['car_27', 'stress_qa', 'vol', 'mom',
            'lnmve', 'bm', 'UE', 'POSWORDS', 'NEGWORDS',
            'symbol', 'time']

regress_data = car_2_7.dropna(subset=reg_vars).copy()
print(f"\nSample: {len(regress_data)} obs | {regress_data['symbol'].nunique()} firms")

# ─── MODELS ───
m1 = pf.feols(
    'car_27 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

m2 = pf.feols(
    'car_27 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

m3 = pf.feols(
    'car_27 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | time',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

m4 = pf.feols(
    'car_27 ~ stress_qa + vol + mom + lnmve + bm + UE + POSWORDS + NEGWORDS | symbol + time',
    data=regress_data,
    vcov={'CRV1': 'symbol + time'}
)

pf.etable([m1, m2, m3, m4])


Sample: 7493 obs | 472 firms


/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/yiqisun/Documents/Thesis/NLP project/venv/lib/python3.11/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


GT(_tbl_data=   __index_level_0__ __index_level_1__                          0  \
0               coef         stress_qa   0.000194 <br> (0.000676)   
1               coef               vol         0.125 <br> (0.126)   
2               coef               mom         0.012 <br> (0.006)   
3               coef             lnmve  -0.000677 <br> (0.000838)   
4               coef                bm         0.006 <br> (0.004)   
5               coef                UE         0.001 <br> (0.001)   
6               coef          POSWORDS        -0.011 <br> (0.093)   
7               coef          NEGWORDS          0.206 <br> (0.25)   
8               coef         Intercept          0.002 <br> (0.01)   
9                 fe              time                          -   
10                fe            symbol                          -   
11                fe           symbol                           -   
12                fe              time                          -   
13             stats      Observations                      7,493   
14             stats                R²                      0.007   

                           1                          2  \
0   0.000541 <br> (0.000902)    0.00043 <br> (0.000709)   
1        -0.018 <br> (0.188)          0.197 <br> (0.15)   
2         0.005 <br> (0.007)          0.01 <br> (0.006)   
3        -0.003 <br> (0.005)  -0.000542 <br> (0.000803)   
4         0.012 <br> (0.009)         0.006 <br> (0.003)   
5         0.002 <br> (0.001)      0.000931 <br> (0.001)   
6        -0.032 <br> (0.164)         0.002 <br> (0.098)   
7         0.118 <br> (0.247)         0.279 <br> (0.222)   
8                                                         
9                          -                          -   
10                         x                          -   
11                         -                          -   
12                         -                          x   
13                     7,491                      7,493   
14                     0.068                      0.023   

                           3  
0   0.000865 <br> (0.000917)  
1         0.095 <br> (0.252)  
2         0.003 <br> (0.007)  
3        -0.003 <br> (0.006)  
4          0.014 <br> (0.01)  
5         0.001 <br> (0.001)  
6         -0.02 <br> (0.166)  
7         0.214 <br> (0.221)  
8                             
9                          x  
10                         -  
11                         x  
12                         -  
13                     7,491  
14                     0.085  , _body=<great_tables._gt_data.Body object at 0x16af96b50>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x16b966650>, _spanners=Spanners([SpannerInfo(spanner_id='car_27', spanner_level=1, spanner_label='car_27', spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1617979d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x110a6f6d0>, _source_notes=['Format of coefficient cell: Coefficient   (Std. Error)'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at

In [71]:
m4.summary()

###

Estimation:  OLS
Dep. var.: car_27, Fixed effects: symbol + time
sample: None = all
Inference:  CRV1
Observations:  7491

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| stress_qa     |      0.001 |        0.001 |     0.943 |      0.356 | -0.001 |   0.003 |
| vol           |      0.095 |        0.252 |     0.376 |      0.711 | -0.428 |   0.618 |
| mom           |      0.003 |        0.007 |     0.455 |      0.653 | -0.011 |   0.017 |
| lnmve         |     -0.003 |        0.006 |    -0.439 |      0.665 | -0.015 |   0.010 |
| bm            |      0.014 |        0.010 |     1.466 |      0.157 | -0.006 |   0.034 |
| UE            |      0.001 |        0.001 |     0.963 |      0.346 | -0.001 |   0.003 |
| POSWORDS      |     -0.020 |        0.166 |    -0.117 |      0.908 | -0.364 |   0.325 |
| NEGWORDS      |      0.214 |        0.221 |     0.969 |      

In [54]:
car_2_7.columns

Index(['symbol', 'year', 'quarter', 'call_date', 'stress_pr', 'stress_qa',
       'stress_whole_x', 'car_07', 'stress_whole_y', 'car_01', 'vol', 'mom',
       'POSWORDS', 'NEGWORDS', 'lnmve', 'bm', 'atq', 'UE', 'actual_eps',
       'median_forecast', 'numest'],
      dtype='str')

In [55]:
car_2_7['stress_pr'].mean()

np.float64(2.032454009024644)

In [56]:
car_2_7['stress_qa'].mean()

np.float64(2.5295880149812735)